In [1]:
import os
import shutil
import random
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping


2026-02-17 20:57:36.095945: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-17 20:57:36.096279: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-17 20:57:36.154887: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-17 20:57:38.646840: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To tur

In [2]:
CARPETA_DATOS = "train"
TAMAÑO_IMG = 150
BATCH_SIZE = 32
EPOCAS = 10

MAX_IMAGENES = 1000 

In [3]:
DATASET_PATH = ("train")

for conjunto in ['train', 'validation', 'test']:
    for animal in ['cats', 'dogs']:
        os.makedirs(f"datos_rapido/{conjunto}/{animal}", exist_ok=True)

animales = {'cat': 'cats', 'dog': 'dogs'}

for carpeta_orig, carpeta_nueva in animales.items():
    ruta = os.path.join(CARPETA_DATOS, carpeta_orig)
    archivos = [f for f in os.listdir(ruta) if f.endswith('.jpg')]
    
    # TOMAR SOLO LAS PRIMERAS IMÁGENES (más rápido)
    archivos = archivos[:MAX_IMAGENES]
    random.shuffle(archivos)
    
    # Dividir: 80% train, 10% validation, 10% test
    limite_train = int(len(archivos) * 0.8)
    limite_val = int(len(archivos) * 0.9)
    
    conjuntos = {
        'train': archivos[:limite_train],
        'validation': archivos[limite_train:limite_val],
        'test': archivos[limite_val:]
    }
    
    for conjunto, lista in conjuntos.items():
        for archivo in lista:
            origen = os.path.join(ruta, archivo)
            destino = f"datos_rapido/{conjunto}/{carpeta_nueva}/{archivo}"
            if not os.path.exists(destino):
                shutil.copy2(origen, destino)
    
    print(f"{carpeta_orig}: {len(conjuntos['train'])} train, {len(conjuntos['validation'])} val, {len(conjuntos['test'])} test")



cat: 800 train, 100 val, 100 test
dog: 800 train, 100 val, 100 test


In [4]:
train_gen = ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True  
)

val_gen = ImageDataGenerator(rescale=1./255)

datos_train = train_gen.flow_from_directory(
    'datos_rapido/train',
    target_size=(TAMAÑO_IMG, TAMAÑO_IMG),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

datos_val = val_gen.flow_from_directory(
    'datos_rapido/validation',
    target_size=(TAMAÑO_IMG, TAMAÑO_IMG),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

datos_test = val_gen.flow_from_directory(
    'datos_rapido/test',
    target_size=(TAMAÑO_IMG, TAMAÑO_IMG),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

print(f" Total imágenes: {datos_train.samples + datos_val.samples + datos_test.samples}")


Found 1600 images belonging to 2 classes.
Found 200 images belonging to 2 classes.
Found 200 images belonging to 2 classes.
 Total imágenes: 2000


# Crear generadores de imágenes

In [5]:
# Solo normalización para validación y test
val_gen = ImageDataGenerator(rescale=1./255)

# Cargar datos
datos_train = train_gen.flow_from_directory(
    'datos/train',
    target_size=(TAMAÑO_IMG, TAMAÑO_IMG),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

Found 17906 images belonging to 2 classes.


In [6]:
datos_val = val_gen.flow_from_directory(
    'datos/validation', 
    target_size=(TAMAÑO_IMG, TAMAÑO_IMG),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

datos_test = val_gen.flow_from_directory(
    'datos/test',
    target_size=(TAMAÑO_IMG, TAMAÑO_IMG), 
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

print(f"Imágenes de entrenamiento: {datos_train.samples}")
print(f"Imágenes de validación: {datos_val.samples}")
print(f"Imágenes de prueba: {datos_test.samples}")

Found 4020 images belonging to 2 classes.
Found 4010 images belonging to 2 classes.
Imágenes de entrenamiento: 17906
Imágenes de validación: 4020
Imágenes de prueba: 4010


# Construir el modelo CNN

In [7]:
modelo = Sequential()

modelo.add(Conv2D(32, (3,3), activation='relu', input_shape=(TAMAÑO_IMG, TAMAÑO_IMG, 3)))
modelo.add(MaxPooling2D(2,2))

modelo.add(Conv2D(64, (3,3), activation='relu'))
modelo.add(MaxPooling2D(2,2))

modelo.add(Conv2D(128, (3,3), activation='relu'))
modelo.add(MaxPooling2D(2,2))

# Capas densas más pequeñas
modelo.add(Flatten())
modelo.add(Dense(128, activation='relu'))  # 128 en vez de 512
modelo.add(Dropout(0.5))
modelo.add(Dense(2, activation='softmax'))

modelo.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(f" Parámetros: {modelo.count_params():,} (modelo simple)")

/usr/local/python/3.12.1/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-02-17 20:59:27.358923: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


 Parámetros: 4,828,610 (modelo simple)


 # Entrenar el modelo

In [8]:
os.makedirs('modelos_rapido', exist_ok=True)

# Solo early stopping 
callback = EarlyStopping(patience=3, restore_best_weights=True)

historial = modelo.fit(
    datos_train,
    epochs=EPOCAS,
    validation_data=datos_val,
    callbacks=[callback],
    verbose=1
)


Epoch 1/10
560/560 ━━━━━━━━━━━━━━━━━━━━ 284s 503ms/step - accuracy: 0.6434 - loss: 0.6196 - val_accuracy: 0.7336 - val_loss: 0.5344
Epoch 2/10
560/560 ━━━━━━━━━━━━━━━━━━━━ 298s 531ms/step - accuracy: 0.7426 - loss: 0.5214 - val_accuracy: 0.7744 - val_loss: 0.4812
Epoch 3/10
560/560 ━━━━━━━━━━━━━━━━━━━━ 279s 498ms/step - accuracy: 0.7901 - loss: 0.4529 - val_accuracy: 0.7933 - val_loss: 0.4437
Epoch 4/10
560/560 ━━━━━━━━━━━━━━━━━━━━ 280s 501ms/step - accuracy: 0.8208 - loss: 0.4043 - val_accuracy: 0.8271 - val_loss: 0.3810
Epoch 5/10
560/560 ━━━━━━━━━━━━━━━━━━━━ 285s 509ms/step - accuracy: 0.8387 - loss: 0.3646 - val_accuracy: 0.8425 - val_loss: 0.3662
Epoch 6/10
560/560 ━━━━━━━━━━━━━━━━━━━━ 336s 534ms/step - accuracy: 0.8603 - loss: 0.3249 - val_accuracy: 0.8450 - val_loss: 0.3474
Epoch 7/10
560/560 ━━━━━━━━━━━━━━━━━━━━ 299s 533ms/step - accuracy: 0.8802 - loss: 0.2865 - val_accuracy: 0.8475 - val_loss: 0.3520
Epoch 8/10
560/560 ━━━━━━━━━━━━━━━━━━━━ 301s 537ms/step - accuracy: 0.8929 -

In [10]:
# Evaluar

perdida, precision = modelo.evaluate(datos_test)

print(f"\n RESULTADO RÁPIDO:")
print(f"Precisión: {precision*100:.1f}%")

if precision > 0.75:
    print(" ¡Excelente para modelo rápido!")
elif precision > 0.65:
    print(" Buen resultado para modelo simple")
else:
    print(" Modelo básico, pero funciona")

# Guardar modelo ligero
modelo.save('modelos_rapido/modelo_rapido.h5')
print(" Modelo guardado en: modelos_rapido/modelo_rapido.h5")
print(" Entrenamiento rápido completado!")


126/126 ━━━━━━━━━━━━━━━━━━━━ 23s 178ms/step - accuracy: 0.8621 - loss: 0.3216



 RESULTADO RÁPIDO:
Precisión: 86.2%
 ¡Excelente para modelo rápido!
 Modelo guardado en: modelos_rapido/modelo_rapido.h5
 Entrenamiento rápido completado!
